## Business Intelligence Analyst Agent

This notebook demonstrates how to build an AI-powered Business Intelligence (BI) Analyst Agent using LangChain.

The agent combines a large language model with multiple specialized tools to answer business and sales-related questions. It can query an internal SQLite sales database, generate SQL from natural-language questions, execute read-only queries, perform currency conversions, search external sources for market information, and maintain conversational context across multiple user requests.

The notebook uses a sample sales dataset generated programmatically and stores the data in a SQLite database. The generated data is then exposed to the agent through SQL-generation and query-execution tools.

#### High level overview

```text

        User Question
            |
            v
        BI Analyst Agent
            |
            +--------------------+
            |                    |
            v                    v
        Internal Data          External Data
            |                    |
            v                    v
        SQL Generator        Tavily Web Search
            |
            v
        Read-only SQL Executor
            |
            v
        SQLite Sales Database
            |
            v
        Business Analysis
            |
            +---- Currency Converter
            |
            v
        Final BI Insight
```

The agent acts as the orchestration layer. Instead of manually deciding which function to call, the language model determines which available tool is appropriate for the user's request.

### Model Selection and Cost

* This notebook uses the GPT-5-Nano model as the underlying LLM for the BI Analyst Agent. 
* GPT-5 Nano is a lightweight and cost-efficient model, making it well suited for experimenting without incurring significant API costs.
* For this notebook's experimentation and sample workload, the total model usage costed me approximately $0.03.

In [ ]:
import os
from dotenv import load_dotenv
from datetime import date, timedelta
import random
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from pydantic import BaseModel
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
import sqlite3
from langchain_tavily import TavilySearch
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langgraph.checkpoint.memory import InMemorySaver
import requests
from rich import print
import sys
sys.path.append('..')
from utils.helper import print_messages

load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not os.getenv("TAVILY"):
    raise ValueError("TAVILY environment variable is not set.")
TAVILY = os.getenv("TAVILY")

llm = ChatOpenAI(model="gpt-5-nano")

#### Create the SQLite Sales Database
* This block establishes a connection to the SQLite database located at: data/sales.db
* A sales table is created if it does not already exis

**This database represents the internal data source that the BI Analyst Agent will analyze.**

In [3]:
connection = sqlite3.connect("data/sales.db")

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS sales (
        order_id INTEGER PRIMARY KEY,
        company TEXT,
        category TEXT,
        product_name TEXT,
        country TEXT,
        order_date Date,
        cost_price REAL,
        selling_price REAL,
        quantity INTEGER,
        currency TEXT
    )
""")

#### Insert records to database table.
* The function takes all fields required to create a sales transaction and inserts the record into the SQLite sales table.
* This function is used to populate the sample database.

In [ ]:
def insert_sales_record_in_database(order_id: int, company: str, category: str, product_name: str, country: str, order_date: str, cost_price: float, selling_price: float, quantity: int, currency: str) -> str:
    """
    Insert a new sales record into the sales database.
    Use this tool when the user wants to add sales data.
    """
    connection = sqlite3.connect("data/sales.db")
    cursor = connection.cursor()
    cursor.execute("""
        INSERT INTO sales (order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency))
    connection.commit()
    connection.close()
    return "Sales record inserted successfully."

#### Define Sample Companies and Products
* This defines the synthetic business data used by the agent to perform analysis on.
* A list of companies is created, including retailers and e-commerce organizations.

In [ ]:
companies = [
    "Amazon",
    "Flipkart",
    "Myntra",
    "Nykaa",
    "Croma",
    "Noon",
    "EMart",
    "Walmart",
    "Snapdeal",
    ]

products = {
    "Electronics": [
        ("Apple iPhone 15", 55000, 70000),
        ("Samsung Galaxy S24", 45000, 60000),
        ("Sony Headphones", 5000, 8000),
        ("Dell Laptop", 45000, 60000),
        ("Boat Smart Watch", 1500, 3000),
    ],

    "Fashion": [
        ("Nike Running Shoes", 2500, 5000),
        ("Levis Jeans", 1200, 2500),
        ("Allen Solly Shirt", 800, 1800),
        ("Puma T-Shirt", 600, 1500),
        ("Adidas Jacket", 3000, 6000),
    ],

    "Beauty": [
        ("Lakme Lipstick", 300, 600),
        ("Maybelline Mascara", 250, 550),
        ("Loreal Shampoo", 300, 700),
        ("The Ordinary Serum", 500, 1200),
        ("Mamaearth Face Wash", 200, 450),
    ],

    "Home Appliances": [
        ("Philips Mixer Grinder", 2500, 4500),
        ("LG Microwave Oven", 8000, 12000),
        ("Samsung Refrigerator", 25000, 40000),
        ("Prestige Cooker", 1000, 2000),
        ("Havells Fan", 1500, 3000),
    ]
}


countries = ["India", "UAE", "USA", "UK", "Singapore"]


def generate_sales_orders(number_of_records=100):
    sales_orders = []
    start_date = date(2023, 1, 1)

    for order_id in range(1, number_of_records + 1):
        company = random.choice(companies)
        category = random.choice(list(products.keys()))
        product_name, cost_range, selling_range = random.choice(products[category])
        quantity = random.randint(1, 10)
        cost_price = round(random.uniform(cost_range * 0.8, cost_range * 1.1), 2)
        selling_price = round(random.uniform(selling_range * 0.9, selling_range * 1.2), 2)
        order_date = start_date + timedelta(days=random.randint(0, 1000))
        country = random.choice(countries)
        sales_orders.append((order_id, company, category, product_name, country, order_date.isoformat(), cost_price, selling_price, quantity, 'USD'))
    return sales_orders


# Generate data
sales_data = generate_sales_orders(200)

# Print first 5 records
for row in sales_data[:5]:
    print(row)

(1, 'Amazon', 'Beauty', 'Lakme Lipstick', 'UK', '2024-04-18', 320.19, 673.01, 9, 'USD')

(2, 'Croma', 'Fashion', 'Nike Running Shoes', 'USA', '2024-06-03', 2277.31, 5674.73, 1, 'USD')

(3, 'Walmart', 'Beauty', 'Mamaearth Face Wash', 'UAE', '2023-02-25', 187.88, 490.85, 8, 'USD')

(4, 'Walmart', 'Beauty', 'Loreal Shampoo', 'India', '2025-04-22', 257.53, 781.86, 7, 'USD')

(5, 'EMart', 'Home Appliances', 'Havells Fan', 'Singapore', '2024-02-15', 1218.55, 2742.51, 1, 'USD')

#### 9. Insert Generated Sales Data into the Database

In [ ]:
for order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency in sales_data:
    result = insert_sales_record_in_database(
        order_id=order_id,
        company=company,
        category=category,
        product_name=product_name,
        country=country,
        order_date=order_date,
        cost_price=cost_price,
        selling_price=selling_price,
        quantity=quantity,
        currency=currency
    )


#### Database Schema Inspection
Function dynamically inspects the SQLite database and retrieves information about the available tables and columns.

In [64]:
def get_database_schema():
    connection = sqlite3.connect("data/sales.db")
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    schema = []

    for (table_name,) in tables:
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        schema.append(f"Table: {table_name}")
        for column in columns:
            schema.append(f"  - {column[1]} ({column[2]})")
    cursor.close()
    return "\n".join(schema)

In [8]:
get_database_schema()

'Table: sales\n  - order_id (INTEGER)\n  - company (TEXT)\n  - category (TEXT)\n  - product_name (TEXT)\n  - country (TEXT)\n  - order_date (Date)\n  - cost_price (REAL)\n  - selling_price (REAL)\n  - quantity (INTEGER)\n  - currency (TEXT)'

#### Natural-Language-to-SQL Tool
* Its purpose is to translate a user's natural-language business question into a valid SQLite SQL query.
* For example, a user might ask:
    > What was the total number of sales per country in 2023?

Instead of requiring user to write SQL manually, this tool provides SQL based on user question.
* The prompt explicitly instructs the model to:
    * Generate valid SQLite SQL.
    * Use only tables and columns that exist.
    * Avoid inventing database objects.
    * Return only SQL.
    * Avoid Markdown code fences.
    * Generate only SELECT queries.

**This creates an important abstraction layer between the business user and the database.**

In [ ]:
@tool
def generate_sql_query(question: str) -> str:
    """
    Convert a user's natural-language question into a SQL query.

    Use this tool when the user asks a question about data
    stored in the database.
    """
    schema = get_database_schema()

    prompt = f"""
        You are an expert SQL query generator for SQLite3 dialect.
        Convert the user's natural-language question into a SQL query.
        Database schema:
        {schema}

        User question:
        {question}

        Rules:
        1. Generate valid SQLite SQL.
        2. Use only tables and columns present in the schema.
        3. Do not invent tables or columns.
        4. Return ONLY the SQL query.
        5. Do not include markdown code fences.
        6. Only generate SELECT queries.

    SQL:
    """

    response = llm.invoke(prompt)

    return response.content.strip()

**Test Tool: Generate a SQL Query from a Business Question**

In [ ]:
query = generate_sql_query.invoke("What was the total number of sales per country in 2023?")
print(query)

SELECT country,
       COUNT(*) AS total_sales
FROM sales
WHERE strftime('%Y', order_date) = '2023'
GROUP BY country
ORDER BY total_sales DESC;

#### Read-Only Sales Query Tool
* This function executes SQL queries against the sales database.
* BI Analyst Agent can use it when answering questions about internal sales data.
* Includes a basic guardrail mechanism that checks the SQL statement for potentially destructive operations such as delete, drop etc
> Note: This is a basic demonstration-level safety check. 
> Production implementations should use stronger SQL parsing, database permissions, read-only database credentials, query validation, and other security controls.

In [45]:
@tool
def run_sales_query(query: str) -> str:
    """
    Use this tool to execute a read-only SQL query against the sales database.
    This tool is intended for analyzing sales data, including questions about sales,
    revenue, products, trends, over time, and other related metrics.
    Args:
        query (str): A read-only SQL query in SQLite3 dialect.
    """
    connection = sqlite3.connect("data/sales.db") 
    cursor = connection.cursor()
    # Basic safety for the demo
    forbidden = ["insert", "update", "delete", "drop", "alter", "truncate" ]

    normalized = query.lower()

    if any(word in normalized for word in forbidden):
        return "Only read-only SQL queries are allowed."

    try:
        result = cursor.execute(query)
        rows = result.fetchall()
        connection.close()
        return rows
    except Exception as e:
        return f"SQL error: {e}"

In [ ]:
res = run_sales_query.invoke(query)
res

[('Singapore', 20), ('UK', 15), ('UAE', 15), ('India', 13), ('USA', 9)]

#### Initialize the Web Search Tool
* This web-search capability allows the agent to supplement information from external sources.
* This is particularly useful for questions involving:
    * Competitor performance.
    * Market trends.
    * External company metrics.
    * Current market conditions.

In [ ]:
web_search = TavilySearch(max_results=1, tavily_api_key=TAVILY)

#### Add currency Converter Tool
* The tool converts an amount from one currency to another using exchange-rate information obtained from the Frankfurter API.
* This tool allows the BI Agent to answer questions where users want financial analysis presented in a different currency.
* For example, a user can ask the agent to provide a sales analysis in INR rather than USD.

In [48]:
@tool
def currency_converter(amount: float, base_currency: str, quote_currency: str) -> float:
    """
    Convert an amount from one currency to another used when the user asks about sales in different currencies.
    Or want to convert the analysis in specific currencies.

    Args:
        amount: Amount to convert.
        base_currency: Three-letter source currency code, e.g. USD.
        quote_currency: Three-letter target currency code, e.g. SGD.
    Returns:
        Converted amount.
    """

    url = "https://api.frankfurter.dev/v2/rates"

    params = {
        "base": base_currency.upper(),
        "quotes": quote_currency.upper()
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    rate = data[0]["rate"]
    converted_amount = amount * rate

    return round(converted_amount, 2)

**Test the Currency Conversion Tool**

In [49]:
converted_amount = currency_converter.invoke(input={"amount": 10, "base_currency": "USD", "quote_currency": "INR"})
print(converted_amount)

952.1

#### User Context Model
* User context is required to include only allowed/permissive tools to the user based on their role.
* It contains: 
    * **user_id** — identifies the user or conversation participant.
    * **role** — describes the user's business role.
* Example roles in this notebook include:
    * bussiness_analyst: Have access to all the tools. 
    * market_analyst: Have access to web_search & currency_converter tools.

In [50]:
class UserContext(BaseModel):
    user_id: str
    role: str

#### Middleware: Dynamically Control Tools Based on User Role
* This middleware introduces role-based tool access to the BI Analyst Agent.
* Instead of exposing every available tool to every user, the middleware examines which tools the agent is allowed to access.

The middleware effectively implements the following flow:
```text

                    User Request
                         |
                         v
                  User Context
                         |
                         v
                  Read user role
              ┌──────────┴──────────┐
              │                     │
              v                     v
     role == business_analyst   Other role
              |                     |
              v                     v
       Full tool access       Restricted tools
              │              ┌──────┴──────┐
              │              v             v
              │       currency_converter web_search
              └──────────────┬──────────────┘
                             v
                         LLM Agent
```

In [ ]:
@wrap_model_call 
def gate_agent_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose currency_converter and web_search to users who are NOT business analysts."""
    user_role = request.context.role
    if user_role != "business_analyst":
        allowed = [t for t in request.tools if t.name in ["currency_converter", "web_search"]]
        request = request.override(tools=allowed)
    return handler(request)

### Create BI Analyst Agent
The agent is provided with:
* **Four tools**:  
    1. generate_sql_query
    2. run_sales_query
    3. web_search
    4. currency_converter
* **system_prompt**
* **context_schema**: A user context schema
* **checkpointer**: It allows the agent to maintain state between previous interactions.
* **middleware**: Dynamically tool loading

In [ ]:
agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[
        generate_sql_query,
        run_sales_query,
        web_search,
        currency_converter
    ],
    system_prompt="""You are an enterprise Business Analyst Agent.
Please utilize all available tools to assist the user in analyzing internal sales data and market trends.
Don't overcomplicate things by asking follow up questions, just provide the analysis what you feel is right.""",
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
    middleware=[gate_agent_tools]
)

**Create Business Analyst User Context**
* The context contains a `user identifier` and the user's `role`.
* A configurable `thread_id` is also created for the conversation.
* The thread identifier is important for maintaining conversational state when the same user continues asking related questions.
* This allows subsequent requests to be associated with the same analytical conversation as demonstrated next in Notebook.

In [ ]:
context = UserContext(user_id="user_123", role="business_analyst")
business_analyst = {"configurable": {"thread_id": "user_12345"}}

**First Interaction of business_analyst with Agent**
* Allowing business users `user_id=user_12345` to interact with structured data using natural language.
* The query required agent to get the information from a database.

In [ ]:
result = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the revenue by country per product category?")]},
        config=business_analyst,
        context=context
)

**Below is the generated output with the first agent interaction from BA user**
* It made tool to `generate_sql_query` which returned the content with **Generated SQL query string from user natural language question**
* It made another tool call to `run_sales_query` which executed the generated query agains our sales database, and returned with the results.
* Then these results are provided to LLM that curated the results and returned `AIMessage` with formatted way, along with **Key Insights**.

In [ ]:
print_messages(result)

HumanMessage: What is the revenue by country per product category?

================================================================================

AIMessage:

Tool Call 1 Name: generate_sql_query
Tool Call 1 Args: {'question': 'What is the revenue by country per product category?'}

================================================================================

...

================================================================================

ToolMessage: [["India", "Beauty", 32200.64], ["India", "Electronics", 3003628.65], ["India", "Fashion", 138319.85],
["India", "Home Appliances", 1433838.9100000001], ["Singapore", "Beauty", 39750.65], ["Singapore", "Electronics", 
1639871.63], ["Singapore", "Fashion", 232864.32], ["Singapore", "Home Appliances", 576270.5], ["UAE", "Beauty", 
22927.86], ["UAE", "Electronics", 1043251.21], ["UAE", "Fashion", 223805.27000000002], ["UAE", "Home Appliances", 
868851.14], ["UK", "Beauty", 38658.28], ["UK", "Electronics", 1763089.65], ["UK", "Fashion", 202080.50999999998], 
["UK", "Home Appliances", 947513.01], ["USA", "Beauty", 59598.01], ["USA", "Electronics", 103487.72], ["USA", 
"Fashion", 207024.81], ["USA", "Home Appliances", 706147.08]]

================================================================================

AIMessage: Here is the revenue by country broken down by product category (in your reporting currency, likely USD),
plus totals.

India
- Electronics: 3,003,628.65
- Fashion: 138,319.85
- Beauty: 32,200.64
- Home Appliances: 1,433,838.91
- Total: 4,607,988.05

Singapore
- Electronics: 1,639,871.63
- Fashion: 232,864.32
- Beauty: 39,750.65
- Home Appliances: 576,270.50
- Total: 2,488,757.10

UAE
- Electronics: 1,043,251.21
- Fashion: 223,805.27
- Beauty: 22,927.86
- Home Appliances: 868,851.14
- Total: 2,158,835.48

United Kingdom
- Electronics: 1,763,089.65
- Fashion: 202,080.51
- Beauty: 36,658.28
- Home Appliances: 947,513.01
- Total: 2,949,341.45

United States
- Electronics: 103,487.72
- Fashion: 207,024.81
- Beauty: 59,598.01
- Home Appliances: 706,147.08
- Total: 1,076,257.62

Key insights
- Electronics is the largest revenue driver in most countries (India, Singapore, UAE, UK). In the USA, Home 
Appliances is the leading category (706k) followed by Fashion (207k) and Electronics (103k).
- By total revenue, India leads (≈4.61M), followed by the UK (≈2.95M), Singapore (≈2.49M), UAE (≈2.16M), and the 
USA (≈1.08M).

If you’d like, I can generate a CSV pivot or a chart (e.g., stacked bars by country) for quick visualization.

================================================================================

**Second Interaction of business_analyst with Agent**
* Asking Agent little complex question and provide information from database.
* As we are using the same `business_analyst` config, agent will maintain a conversation history for this user leveraging `thread_id`

In [ ]:
result_3 = agent.invoke(
    {"messages": [
        HumanMessage(content="I would like to analyze year over year revenue, profit/loss trends in percentages also")]},
        config=business_analyst,
        context=context
)

**Below is the generated output with the Second agent interaction from BA user**
* This time showing only the Last message we received from LLM `AIMessage`
* We got the response a expected and Agent is actually able to provide All the required information user Asked.
* Agent Also provided the **Below Key Insights** from the results.
    * 2024 saw a sharp revenue decline compared with 2023 (-19.7%).
    * 2025 revenue rebounded strongly (+18.0% vs 2024)

>**In the end, we'll see the full conversation history of `business_analyst` user.**

In [56]:
print(result_3["messages"][-1].content)

Here are the year-over-year (YoY) trends for revenue and profitability (gross profit and gross margin). I’m using 
gross profit as the profitability proxy since your data doesn’t include operating expenses, taxes, etc.

Year 2023
- Revenue: 4,828,353.07
- Gross Profit: 1,963,017.45
- Gross Margin: 40.66%
- YoY revenue: N/A (base year)

Year 2024
- Revenue: 3,878,038.26
- Gross Profit: 1,520,274.45
- Gross Margin: 39.20%
- YoY revenue change vs 2023: -19.68%
- YoY gross margin change: -3.58 percentage points

Year 2025
- Revenue: 4,576,788.37
- Gross Profit: 1,796,271.51
- Gross Margin: 39.25%
- YoY revenue change vs 2024: +18.02%
- YoY gross margin change: +0.12 percentage points

Key insights
- 2024 saw a sharp revenue decline compared with 2023 (-19.7%), accompanied by a margin squeeze (gross margin down 
~3.6 pp).
- 2025 revenue rebounded strongly (+18.0% vs 2024), with gross margin stabilizing and showing a small uptick (~0.12
pp). Gross margin remains around 39–40%.
- If net profitability is a criterion, you’ll want to incorporate operating expenses, taxes, and any other costs to
compute net profit and net margin. I can compute a net-profit YoY view if you provide that data.

Next steps (optional)
- Provide a CSV or dashboard-ready export for these yearly figures.
- Break down YoY trends by country or product category to identify drivers behind the 2024 dip and 2025 rebound.
- Add a moving-average or CAGR view for longer-term trend analysis (e.g., 3-year CAGR).
- If you want, I can compute net profit margin and net profit YoY once you share operating expenses.

**Third Interaction of business_analyst with Agent**
* Asking Agent convert this analysis / Metric in different currency (INR)
* This query should use `currency_converter` tool that calls Realtime API to currency conversion based on current rate.

In [ ]:
result_4 = agent.invoke(
    {"messages": [
        HumanMessage(content="Can you provide this analysis in INR instead of USD?")]},
        config=business_analyst,
        context=context
)

**Metrics below are converted to INR**
* Showing only the Last message we received from LLM `AIMessage`

> This Agent invocation make `currency_converter` tool calls **22 Times** to convert each number in previous message in INR.<br>
> We'll see this conversation and tool calls in end of our notebook execution.

In [59]:
print(result_4["messages"][-1].content)

Converted to INR (approximate, using a blended rate around 95.2 INR per 1 USD based on the dataset).

Revenue by country per product category (INR)

India
- Electronics: 285,975,483.77
- Fashion: 13,169,432.92
- Beauty: 3,065,822.93
- Home Appliances: 136,515,802.62
- Total: 438,726,542.24

Singapore
- Electronics: 156,132,177.89
- Fashion: 22,171,011.91
- Beauty: 3,784,659.39
- Home Appliances: 54,866,714.30
- Total: 236,954,563.49

UAE
- Electronics: 99,327,947.70
- Fashion: 21,308,499.76
- Beauty: 2,182,961.55
- Home Appliances: 82,723,317.04
- Total: 205,542,726.05

United Kingdom
- Electronics: 167,863,765.58
- Fashion: 19,240,085.36
- Beauty: 3,490,234.84
- Home Appliances: 90,212,713.68
- Total: 280,806,799.46

United States
- Electronics: 9,853,065.82
- Fashion: 19,710,832.16
- Beauty: 5,674,326.53
- Home Appliances: 67,232,263.49
- Total: 102,470,487.99

Year-over-year revenue and profitability in INR

Revenue (INR)
- 2023: approximately 459,707,495.79
- 2024: approximately 369,228,022.73
- 2025: approximately 435,756,020.71
- YoY revenue changes:
  - 2024 vs 2023: about -19.68%
  - 2025 vs 2024: about +18.01%

Profitability (gross) in INR
- Gross Profit (INR):
  - 2023: approximately 186,898,891.42
  - 2024: approximately 144,745,330.38
  - 2025: approximately 171,022,990.47
- Gross Margin (as % of revenue, INR basis):
  - 2023: 40.66%
  - 2024: 39.20%
  - 2025: 39.25%
- YoY gross profit changes:
  - 2024 vs 2023: about -22.57%
  - 2025 vs 2024: about +18.16%

Notes and caveats
- FX rate: Values are approximate. I used a blended rate around 95.2 INR per USD based on the provided conversions.
If you have a specific date-based FX rate or currency (e.g., use end-of-quarter rate), I can recompute precisely.
- These figures use gross profit as profitability (no operating expenses, taxes, or interest included). If you 
share operating costs, I can calculate net profit and net margin in INR.
- If you’d like, I can export these figures to CSV, generate charts (e.g., stacked bars by country or line charts 
for YoY trends), or break down YoY trends by country or by category for deeper insights.

**First Interaction of market_analyst with Agent**
* Allowing business users `user_id=user_6789` to interact with external information.
* This user doesn't have access to tools that interacts with internal database and only have access to `currency_converter` and `web_search` tools.
* Below query required agent to get the information from a **Tavily Websearch Tool**.

In [95]:
context = UserContext(user_id="user_6789", role="market_analyst")
market_analyst = {"configurable": {"thread_id": "user_6789"}}

In [96]:
result_5 = agent.invoke(
    {"messages": [
        HumanMessage(content="""Tell in brief summary, dont ask follow-up question, return what you feel is the best answer to the user query.:
        Based on external data sources, which companies are performing best globally in below metrics for 2026 YTD
        Give top company for each metric separately:
        - YoY revenue growth
        - Net profit margin
        """)]},
        config=market_analyst,
        context=context
)

**Below is the Agent's Response**

In [97]:
print(result_5["messages"][-1].content)

Here’s a best-effort, concise read from external data signals for 2026 YTD:

- YoY revenue growth (global): Nvidia (NVDA)
  Why: AI-enabled demand has driven outsized revenue growth for Nvidia across data center and AI GPU markets, often
cited as a standout in year-to-date growth signals among global tech peers.

- Net profit margin (global): Microsoft (MSFT)
  Why: Microsoft consistently posts one of the highest net margins among large global tech names, supported by 
strong profitability in cloud and software segments.

Note: This is a best-effort read from publicly reported signals. A precise, globally definitive top for 2026 YTD 
would require a harmonized data set (quarterly/annual filings across all sectors) and consistent margin 
definitions. If you’d like, I can compile a data-backed ranking from primary sources (filings, presentations) and 
present exact top values.

**Conversation history of market_analyst**
* We are using new `market_analyst` config, agent will maintain a seperate conversation history for this user leveraging `thread_id`
* This time we'll not see the converation we made with Agent from `business_analyst` role.
* In reponse we can see Agent made some calls to `tavily_search` tools with different arguments to get complete information before giving final response.

In [98]:
print_messages(result_5)

HumanMessage: Tell in brief summary, dont ask follow-up question, return what you feel is the best answer to the 
user query.:
        Based on external data sources, which companies are performing best globally in below metrics for 2026 YTD
        Give top company for each metric separately:
        - YoY revenue growth
        - Net profit margin

================================================================================

AIMessage:

Tool Call 1 Name: tavily_search
Tool Call 1 Args: {'query': 'top companies by YoY revenue growth 2026 year-to-date', 'time_range': 'year', 'topic':
'finance'}

Tool Call 2 Name: tavily_search
Tool Call 2 Args: {'query': 'highest net profit margin 2026 year to date', 'time_range': 'year', 'topic': 
'finance'}

...

================================================================================

ToolMessage: {"query": "top net profit margin 2026 YTD globally", "follow_up_questions": null, "answer": null, 
"images": [], "results": [{"url": 
"https://finance.yahoo.com/markets/stocks/articles/robinhood-down-30-2026-while-134819485.html", "title": 
"Robinhood Is Down 30% in 2026 While Interactive Brokers Is Up 35%: Here’s Why", "content": "2. Best 0% APR cards. 
3. Best money market accounts. # Robinhood Is Down 30% in 2026 While Interactive Brokers Is Up 35%: Here’s Why. * 
Robinhood (HOOD) stock fell 30% YTD after Q1 revenue of $1.07B missed consensus by 6%, with cryptocurrency revenue 
collapsing 47% YoY to $134M. * Interactive Brokers (IBKR) stock surged 36% YTD on Q1 revenue of $1.68B and net 
income of $1.17B lifted by 19% commission growth and 17% net interest income expansion to $904M. * Robinhood's 
retail-dependent revenue mix swings with crypto and options engagement, while Interactive Brokers' professional and
global client base compounds through commissions and net interest income on a larger customer equity pool, creating
divergent trajectories in the same industry. The gap reflects how cyclical retail engagement at Robinhood is 
colliding with the steadier compounding of Interactive Brokers' professional, global client base. Revenue of $1.07 
billion missed the $1.14 billion consensus by 6%, and EPS of $0.38 came in shy of the $0.39 estimate, snapping a 
four-quarter beat streak.", "score": 0.32972667, "raw_content": null, "id": "1fda7f-00"}], "response_time": 2.13, 
"request_id": "2b8eb31f-39b0-4929-b9e8-9af2e5213477"}

================================================================================

AIMessage: Here’s a best-effort, concise read from external data signals for 2026 YTD:

- YoY revenue growth (global): Nvidia (NVDA)
  Why: AI-enabled demand has driven outsized revenue growth for Nvidia across data center and AI GPU markets, often
cited as a standout in year-to-date growth signals among global tech peers.

- Net profit margin (global): Microsoft (MSFT)
  Why: Microsoft consistently posts one of the highest net margins among large global tech names, supported by 
strong profitability in cloud and software segments.

Note: This is a best-effort read from publicly reported signals. A precise, globally definitive top for 2026 YTD 
would require a harmonized data set (quarterly/annual filings across all sectors) and consistent margin 
definitions. If you’d like, I can compile a data-backed ranking from primary sources (filings, presentations) and 
present exact top values.

================================================================================

**Fourth Interaction of business_analyst with Agent**
* Asking Agent drill the analysis down by Country Also.
* We didn't provided any context this time, which analysis, metrics, level of analysis etc, just asked to drill the analysis down by country.

In [ ]:
result_6 = agent.invoke(
    {"messages": [
        HumanMessage(content="I would like to further drill the analysis down by country")]},
        config=business_analyst,
        context=context
)

**Below is the Agent's Response**
* Agent responded with analysis drilled further down by country.
* Using `MemorySaver/Checkpointer` agent provided the context from previous interation of `business_analyst` automatically.
* Our previous conversation had `HumanMessage`, `AIMessage` etc, that already had enough information for LLM to understand what user is asking for. 

In [62]:
print(result_6["messages"][-1].content)

Here’s a country-by-country drill-down, with figures converted to INR (using the same USD-to-INR rate used earlier:
95.2 INR per 1 USD). Metrics shown per year: Revenue INR, Gross Profit INR, Gross Margin %, plus YoY changes in 
revenue and gross margin.

India
- 2023: Revenue INR 144,717,919.19; Gross Profit INR 58,235,225.16; Gross Margin 40.24%
- 2024: Revenue INR 107,066,739.98; Gross Profit INR 46,367,363.73; Gross Margin 43.31%
  - YoY Revenue: -26.02%; YoY Gross Margin: +7.62 pp
  - Revenue Delta INR: -37,651,179.22
- 2025: Revenue INR 186,895,803.19; Gross Profit INR 74,779,099.25; Gross Margin 40.01%
  - YoY Revenue: +74.56%; YoY Gross Margin: -7.61 pp
  - Revenue Delta INR: +79,829,063.22

Singapore
- 2023: Revenue INR 87,080,193.98; Gross Profit INR 36,731,139.76; Gross Margin 42.18%
- 2024: Revenue INR 99,039,375.06; Gross Profit INR 33,208,588.39; Gross Margin 33.53%
  - YoY Revenue: +13.73%; YoY Gross Margin: -20.51 pp
  - Revenue Delta INR: +11,959,181.08
- 2025: Revenue INR 50,810,106.87; Gross Profit INR 15,711,431.96; Gross Margin 30.92%
  - YoY Revenue: -48.70%; YoY Gross Margin: -7.78 pp
  - Revenue Delta INR: -48,292,268.19

United Arab Emirates (UAE)
- 2023: Revenue INR 81,780,847.90; Gross Profit INR 34,810,920.54; Gross Margin 42.57%
- 2024: Revenue INR 48,750,767.13; Gross Profit INR 19,039,241.26; Gross Margin 39.05%
  - YoY Revenue: -40.39%; YoY Gross Margin: -8.25 pp
  - Revenue Delta INR: -33,030,080.78
- 2025: Revenue INR 74,989,522.66; Gross Profit INR 32,433,564.24; Gross Margin 43.25%
  - YoY Revenue: +53.82%; YoY Gross Margin: +4.77 pp
  - Revenue Delta INR: +26,238,755.54

United Kingdom (UK)
- 2023: Revenue INR 113,605,313.02; Gross Profit INR 42,406,273.16; Gross Margin 37.33%
- 2024: Revenue INR 61,741,075.59; Gross Profit INR 21,680,031.18; Gross Margin 35.11%
  - YoY Revenue: -45.65%; YoY Gross Margin: -5.93 pp
  - Revenue Delta INR: -51,864,237.43
- 2025: Revenue INR 105,621,317.42; Gross Profit INR 39,923,090.09; Gross Margin 37.80%
  - YoY Revenue: +71.07%; YoY Gross Margin: +7.64 pp
  - Revenue Delta INR: +43,880,241.83

United States (USA)
- 2023: Revenue INR 32,474,938.16; Gross Profit INR 14,694,702.62; Gross Margin 45.25%
- 2024: Revenue INR 52,591,284.59; Gross Profit INR 24,434,903.08; Gross Margin 46.46%
  - YoY Revenue: +61.94%; YoY Gross Margin: +2.68 pp
  - Revenue Delta INR: +20,116,346.43
- 2025: Revenue INR 17,393,502.67; Gross Profit INR 8,157,862.22; Gross Margin 46.90%
  - YoY Revenue: -66.93%; YoY Gross Margin: +0.95 pp
  - Revenue Delta INR: -35,197,781.92

Key takeaways by country
- India shows a strong rebound in 2025 after a dip in 2024, with margin around 40% each year.
- Singapore and UK experience large 2024 dips with partial recoveries in 2025; margins fluctuating notably, 
indicating potential mix shifts or pricing changes.
- UAE stabilized in 2025 with a higher margin, after a soft 2024.
- USA saw a substantial drop in 2025 despite solid performance in 2024; margins remain healthy but revenue 
volatility is high.

Next steps (optional)
- Create country-specific dashboards or CSV exports to enable visual inspection (line charts for revenue and gross 
margin, stacked bars for revenue by product category within each country).
- Drill down by product category within each country to identify drivers of growth or decline.
- If you want net profitability, provide operating expenses so I can compute country-level net profit and net 
margin per year.
- I can also generate CAGR, moving averages, or YoY tables tailored to each country.

Would you like me to export this drill-down to CSV and/or generate per-country charts?

## This is the Whole Conversation of Bussiness Analyst with the Agent

In [93]:
print_messages(result_6)

HumanMessage: What is the revenue by country per product category?

================================================================================

AIMessage:

Tool Call 1 Name: generate_sql_query
Tool Call 1 Args: {'question': 'What is the revenue by country per product category?'}

================================================================================

...

================================================================================

ToolMessage: [["India", 2023, 144717919.192, 58235225.16, 40.24050752328619, null, null, null, null, null, null], 
["India", 2024, 107066739.97600001, 46367363.728000015, 43.30697258401039, 2023, 144717919.192, -37651179.21599998,
-26.01694346230024, 40.24050752328619, 7.620343900855898], ["India", 2025, 186895803.192, 74779099.248, 
40.01111740919011, 2024, 107066739.97600001, 79829063.21599999, 74.56009516484241, 43.30697258401039, 
-7.610449260628198], ["Singapore", 2023, 87080193.984, 36731139.76, 42.18081986214791, null, null, null, null, 
null, null], ["Singapore", 2024, 99039375.064, 33208588.391999997, 33.53069258619651, 2023, 87080193.984, 
11959181.080000002, 13.73352599811315, 42.18081986214791, -20.507252595423882], ["Singapore", 2025, 50810106.872, 
15711431.959999993, 30.92186363547705, 2024, 99039375.064, -48229268.191999994, -48.697064335102965, 
33.53069258619651, -7.7804207116003], ["UAE", 2023, 81780847.904, 34810920.536000006, 42.566103712770825, null, 
null, null, null, null, null], ["UAE", 2024, 48750767.128000006, 19039241.256000005, 39.0542393025541, 2023, 
81780847.904, -33030080.776, -40.38852814875799, 42.566103712770825, -8.25037789202934], ["UAE", 2025, 
74989522.664, 32433564.24, 43.25079436139721, 2024, 48750767.128000006, 26238755.535999995, 53.8222413343928, 
39.0542393025541, 10.745453333074291], ["UK", 2023, 113605313.02400002, 42407273.16000002, 37.328600248688325, 
null, null, null, null, null, null], ["UK", 2024, 61741075.592, 21680031.184, 35.1144371492115, 2023, 
113605313.02400002, -51864237.43200002, -45.65300341282743, 37.328600248688325, -5.931546012241995], ["UK", 2025, 
105621317.424, 39923090.087999985, 37.79832619179998, 2024, 61741075.592, 43880241.831999995, 71.07139195625822, 
35.1144371492115, 7.643263741303482], ["USA", 2023, 32474938.16, 14694702.623999998, 45.24936291364441, null, null,
null, null, null, null], ["USA", 2024, 52591284.592, 24434903.08, 46.46188673572912, 2023, 32474938.16, 
20116346.431999996, 61.94421782387775, 45.24936291364441, 2.6796483839976517], ["USA", 2025, 17393502.672000002, 
8157862.216000002, 46.90177918638837, 2024, 52591284.592, -35197781.92, -66.92702449286467, 46.46188673572912, 
0.946781290138556]]

================================================================================

AIMessage: Here’s a country-by-country drill-down, with figures converted to INR (using the same USD-to-INR rate 
used earlier: 95.2 INR per 1 USD). Metrics shown per year: Revenue INR, Gross Profit INR, Gross Margin %, plus YoY 
changes in revenue and gross margin.

India
- 2023: Revenue INR 144,717,919.19; Gross Profit INR 58,235,225.16; Gross Margin 40.24%
- 2024: Revenue INR 107,066,739.98; Gross Profit INR 46,367,363.73; Gross Margin 43.31%
  - YoY Revenue: -26.02%; YoY Gross Margin: +7.62 pp
  - Revenue Delta INR: -37,651,179.22
- 2025: Revenue INR 186,895,803.19; Gross Profit INR 74,779,099.25; Gross Margin 40.01%
  - YoY Revenue: +74.56%; YoY Gross Margin: -7.61 pp
  - Revenue Delta INR: +79,829,063.22

Singapore
- 2023: Revenue INR 87,080,193.98; Gross Profit INR 36,731,139.76; Gross Margin 42.18%
- 2024: Revenue INR 99,039,375.06; Gross Profit INR 33,208,588.39; Gross Margin 33.53%
  - YoY Revenue: +13.73%; YoY Gross Margin: -20.51 pp
  - Revenue Delta INR: +11,959,181.08
- 2025: Revenue INR 50,810,106.87; Gross Profit INR 15,711,431.96; Gross Margin 30.92%
  - YoY Revenue: -48.70%; YoY Gross Margin: -7.78 pp
  - Revenue Delta INR: -48,292,268.19

United Arab Emirates (UAE)
- 2023: Revenue INR 81,780,847.90; Gross Profit INR 34,810,920.54; Gross Margin 42.57%
- 2024: Revenue INR 48,750,767.13; Gross Profit INR 19,039,241.26; Gross Margin 39.05%
  - YoY Revenue: -40.39%; YoY Gross Margin: -8.25 pp
  - Revenue Delta INR: -33,030,080.78
- 2025: Revenue INR 74,989,522.66; Gross Profit INR 32,433,564.24; Gross Margin 43.25%
  - YoY Revenue: +53.82%; YoY Gross Margin: +4.77 pp
  - Revenue Delta INR: +26,238,755.54

United Kingdom (UK)
- 2023: Revenue INR 113,605,313.02; Gross Profit INR 42,406,273.16; Gross Margin 37.33%
- 2024: Revenue INR 61,741,075.59; Gross Profit INR 21,680,031.18; Gross Margin 35.11%
  - YoY Revenue: -45.65%; YoY Gross Margin: -5.93 pp
  - Revenue Delta INR: -51,864,237.43
- 2025: Revenue INR 105,621,317.42; Gross Profit INR 39,923,090.09; Gross Margin 37.80%
  - YoY Revenue: +71.07%; YoY Gross Margin: +7.64 pp
  - Revenue Delta INR: +43,880,241.83

United States (USA)
- 2023: Revenue INR 32,474,938.16; Gross Profit INR 14,694,702.62; Gross Margin 45.25%
- 2024: Revenue INR 52,591,284.59; Gross Profit INR 24,434,903.08; Gross Margin 46.46%
  - YoY Revenue: +61.94%; YoY Gross Margin: +2.68 pp
  - Revenue Delta INR: +20,116,346.43
- 2025: Revenue INR 17,393,502.67; Gross Profit INR 8,157,862.22; Gross Margin 46.90%
  - YoY Revenue: -66.93%; YoY Gross Margin: +0.95 pp
  - Revenue Delta INR: -35,197,781.92

Key takeaways by country
- India shows a strong rebound in 2025 after a dip in 2024, with margin around 40% each year.
- Singapore and UK experience large 2024 dips with partial recoveries in 2025; margins fluctuating notably, 
indicating potential mix shifts or pricing changes.
- UAE stabilized in 2025 with a higher margin, after a soft 2024.
- USA saw a substantial drop in 2025 despite solid performance in 2024; margins remain healthy but revenue 
volatility is high.

Next steps (optional)
- Create country-specific dashboards or CSV exports to enable visual inspection (line charts for revenue and gross 
margin, stacked bars for revenue by product category within each country).
- Drill down by product category within each country to identify drivers of growth or decline.
- If you want net profitability, provide operating expenses so I can compute country-level net profit and net 
margin per year.
- I can also generate CAGR, moving averages, or YoY tables tailored to each country.

Would you like me to export this drill-down to CSV and/or generate per-country charts?

================================================================================